<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>


# Crypto Foundations
## Chapter 5 – Message Authentication Codes

&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH<br>
The Crypto Engineer | AI-Powered by GPT 5.1.


In this first lab you practise signing an API-style request with HMAC. The example shows how to build a canonical string from method, path, and body, then compute a tag with a shared key so the server can verify both integrity and authenticity before acting on the request.

In [1]:
import hmac
import hashlib

key = b"super-secret-api-key"  # shared between client and server
method = "POST"
path = "/v1/transfer"
body = "{'from': 'alice', 'to': 'bob', 'amount': 10}"

canonical = "|".join([method, path, body]).encode("utf-8")  # canonical representation
tag = hmac.new(key, canonical, hashlib.sha256).hexdigest()  # HMAC over request
print("Request:", canonical)
print("Tag:", tag)

# Verifier recomputes and compares in constant time
check = hmac.new(key, canonical, hashlib.sha256).hexdigest()
print("Valid?", hmac.compare_digest(tag, check))


Request: b"POST|/v1/transfer|{'from': 'alice', 'to': 'bob', 'amount': 10}"
Tag: 8ae7c61710ff91b4475b6fb09a3311709c9b61b9af63a970ad9d5f99927126e1
Valid? True


Next you attach a basic freshness window to your HMAC-protected requests. By including a timestamp in the signed data, you give the server enough information to reject old, replayed messages even if the attacker knows the tag.

In [2]:
import time
import hmac
import hashlib

key = b"another-shared-key"
now = int(time.time())  # current Unix timestamp
window = 300  # seconds of freshness (5 minutes)

payload = f"{now}|/v1/transfer|alice|bob|10".encode("utf-8")
tag = hmac.new(key, payload, hashlib.sha256).hexdigest()

print("Payload:", payload)
print("Tag:", tag)

# Verifier side: check both timestamp window and tag
received_time = now  # try adjusting this value to simulate latency
if abs(time.time() - received_time) <= window:
    check = hmac.new(key, payload, hashlib.sha256).hexdigest()
    print("Fresh and valid?", hmac.compare_digest(tag, check))
else:
    print("Rejected: message outside freshness window")


Payload: b'1770904890|/v1/transfer|alice|bob|10'
Tag: 6c2776c0fae6102fca00eacafbfeaeb1c1e970d3358680fb3daed53ea4256edd
Fresh and valid? True


Finally, you see how MACs can protect both file content and metadata together. This helps prevent "mix and match" attacks where an adversary swaps filenames or versions while keeping the same raw bytes.

In [3]:
import hmac
import hashlib
from pathlib import Path

key = b"file-mac-key"  # symmetric key for file MACs
path = Path("mac_example.txt")
if not path.exists():
    path.write_text("important configuration", encoding="utf-8")

metadata = {"name": path.name, "version": "1"}
meta_bytes = f"{metadata['name']}|{metadata['version']}".encode("utf-8")

h = hmac.new(key, meta_bytes, hashlib.sha256)
with path.open("rb") as f:
    for chunk in iter(lambda: f.read(4096), b""):
        h.update(chunk)  # feed file content
tag = h.hexdigest()

print("Metadata:", metadata)
print("Tag:", tag)


Metadata: {'name': 'mac_example.txt', 'version': '1'}
Tag: 9121b0ad8e37b40558a9b1fe27988ff0b10677f9f089aae2e8747f7013df1ed7


<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>
